# Milestone 4 lab, notebook 3 of 8: Test the SQL, not the data

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| 2. How serious is it | Tell dbt which failures stop the night. Keep the rows a test returns |
| **3. Test the SQL, not the data** (you are here) | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import dbt, prepare_week, show_file, sql

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.


## Step 1: A check that was wrong for a week

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

**Ines Marchetti** manages store `S04`. On Tuesday 25 August her store sent no sales file. A whole day, $587.42, is missing from her bonus.

The check that should notice is "the receipts add up to the till's own total". Imagine you wrote its SQL a little wrong, and for a week it said nothing about `S04`. Nobody would know. A check that is silent looks exactly like a check that passes.

A test is code. Code has bugs. So who tests the test?

## Step 2: Data that is prepared first

Since milestone 2 you know that many checks need the data prepared first. In dbt, prepared data gets its own file. It is called a **model**: a query whose result dbt saves as a table.

This model puts two counts of the same money side by side.

In [2]:
show_file("dbt_project/models/prepared/till_reconciliation.sql")

# dbt_project/models/prepared/till_reconciliation.sql
-- Two counts of the same money, side by side: one row per till per day.
--
--   till_cents    what the till counted itself
--   lines_cents   what the sales lines of that till add up to
--
-- A till with no lines counts as zero, so a missing sales file shows as a difference.
SELECT t.store_id,
       t.till_id,
       t.business_date AS sales_date,
       t.net_total_cents AS till_cents,
       coalesce(l.lines_cents, 0) AS lines_cents,
       coalesce(l.lines_cents, 0) - t.net_total_cents AS difference_cents
FROM {{ source('clean', 'till_totals') }} t
LEFT JOIN (
    SELECT store_id, till_id, sales_date, sum(amount_cents) AS lines_cents
    FROM {{ source('clean', 'sales_lines') }}
    GROUP BY store_id, till_id, sales_date
) l
  ON l.store_id = t.store_id AND l.till_id = t.till_id AND l.sales_date = t.business_date


And here is its result for Ines's Tuesday. dbt built it as the table `prepared.till_reconciliation`.

In [3]:
sql("""
    SELECT * FROM prepared.till_reconciliation
    WHERE store_id = 'S04' AND sales_date = DATE '2026-08-25'
""")

,store_id,till_id,sales_date,till_cents,lines_cents,difference_cents
0,S04,T1,2026-08-25,31289,0,-31289
1,S04,T2,2026-08-25,27453,0,-27453


Two tills, and no lines: `lines_cents` is 0, and the difference is the whole day. That zero is the point. A till with no lines must count as zero, or the missing day would not be a row at all.

Two words in the SQL do that work: `LEFT JOIN`, and `coalesce(..., 0)`.

## Step 3: Get it wrong, on four rows

Suppose you had written a plain `JOIN`. It is the first thing most people type.

Do not test that on tonight's data. Test it on a few rows you type yourself, where you know the answer. Two tills. `T1` has two lines, a sale of $7.00 and a refund of $2.00. `T2` has no lines at all.

In [4]:
FOUR_ROWS = """
    WITH till_totals AS (
        SELECT 'S01' AS store_id, 'T1' AS till_id, DATE '2026-08-24' AS business_date, 500 AS net_total_cents
        UNION ALL SELECT 'S01', 'T2', DATE '2026-08-24', 300
    ), sales_lines AS (
        SELECT 'S01' AS store_id, 'T1' AS till_id, DATE '2026-08-24' AS sales_date, 700 AS amount_cents
        UNION ALL SELECT 'S01', 'T1', DATE '2026-08-24', -200
    )
"""

RIGHT = FOUR_ROWS + """
    SELECT t.till_id, t.net_total_cents AS till_cents, coalesce(l.lines_cents, 0) AS lines_cents
    FROM till_totals t
    LEFT JOIN (SELECT store_id, till_id, sum(amount_cents) AS lines_cents
               FROM sales_lines GROUP BY store_id, till_id) l
      ON l.store_id = t.store_id AND l.till_id = t.till_id
    ORDER BY t.till_id
"""
sql(RIGHT)

,till_id,till_cents,lines_cents
0,T1,500,500
1,T2,300,0


Two rows. `T2` is there, with 0 beside 300. Now the plain `JOIN`.

In [5]:
sql(RIGHT.replace("LEFT JOIN", "JOIN"))

,till_id,till_cents,lines_cents
0,T1,500,500


One row. `T2` is gone. The till that had no lines, the one case this model exists for, is the row the wrong SQL drops. And on a normal night, when every store sends its file, both queries give the same answer. You would never see the bug in the data.

## Step 4: A unit test

What you just did has a name, and dbt can do it for you on every change of the code. A **unit test** is a test of your SQL on a few rows typed by hand, with the answer typed beside them.

This one is in the project. Read it: `given` are the rows, `expect` is the answer.

In [6]:
show_file("dbt_project/models/prepared/prepared.yml", "- name: a_till_with_no_lines_counts_as_zero", 17)

# dbt_project/models/prepared/prepared.yml, from line 22
  - name: a_till_with_no_lines_counts_as_zero
    description: A store that sent no sales file must show as a difference, not as nothing.
    model: till_reconciliation
    given:
      - input: source('clean', 'till_totals')
        rows:
          - {store_id: S01, till_id: T1, business_date: 2026-08-24, net_total_cents: 500}
          - {store_id: S01, till_id: T2, business_date: 2026-08-24, net_total_cents: 300}
      - input: source('clean', 'sales_lines')
        rows:
          - {store_id: S01, till_id: T1, sales_date: 2026-08-24, amount_cents: 700}
          - {store_id: S01, till_id: T1, sales_date: 2026-08-24, amount_cents: -200}
    expect:
      rows:
        - {store_id: S01, till_id: T1, sales_date: 2026-08-24, till_cents: 500, lines_cents: 500, difference_cents: 0}
        - {store_id: S01, till_id: T2, sales_date: 2026-08-24, till_cents: 300, lines_cents: 0, difference_cents: -300}



They are the same rows you just typed. Run the unit tests of the project.

In [7]:
dbt("test", "--select", "test_type:unit")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 2 PASS store_day_status::the_five_answers_of_store_day_status  ...  [PASS in 0.11s]
2 of 2 PASS till_reconciliation::a_till_with_no_lines_counts_as_zero  ...  [PASS in 0.04s]
Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=2


Two pass. The second one, `the_five_answers_of_store_day_status`, tests the model that decides whether a store can be paid. It gives one invented store to each of the five answers the model can give. You meet that model in notebook 6.

Notice what the unit test did **not** read: tonight's sales. It brought its own four rows. (dbt still needs the real tables to exist, to know the type of each column.)

## Step 5: Two kinds of test

Everything you wrote in milestones 1 to 3 was the other kind. dbt calls it a **data test**.

| | Data test | Unit test |
|---|---|---|
| It asks | Is tonight's **data** right? | Is my **SQL** right? |
| It runs on | The real rows in the warehouse | A few rows typed by hand |
| The right answer | Not known. No rows back means "fine" | Typed beside the rows |
| It runs | Every night, at a gate | When somebody changes the code |
| When it fails | The data is bad, or the rule is | The SQL is wrong |

That is why the gates do not run unit tests. Tonight's data cannot break them. Only a change to the code can.

A data test with a bug is silent. A unit test is how you find out before a store manager does.

## What you now know

- Data that must be prepared before a check gets its own file in dbt: a model.
- A **unit test** tests your SQL on a few rows typed by hand, with the answer typed beside them. A data test checks tonight's data.
- The bug that matters is in the rare case: a till with no lines. Type that case by hand.
- Data tests run every night. Unit tests run when the code changes.

**Say it in an interview:** "Our reconciliation model had to count a till with no sales lines as zero, or a missing file would be invisible. I wrote a dbt unit test for exactly that case: two tills, one with no lines. With an inner join the test fails. On normal nights the bug would never show in the data."

**The theory:** [Unit Tests in dbt](https://www.d8loop.com/modules/knowledge-corner/dbt/unit-tests-in-dbt).

Next: [notebook 4](04_a_gate_and_where_it_stands.ipynb). The rules are in dbt. Now put them in the way of the data.